In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q11/annotated/checkpoints/pre_cell_3.pickle

trying: ['tpch_parent']
me:  0
trying: ['DATA_ROOT']
me:  0
trying: ['load_partsupp']
me:  1
trying: ['load_nation']
me:  3
trying: ['supplier']


me:  2
trying: ['nation']
me:  3
trying: ['STORAGE_OPTS']
me:  0
trying: ['partsupp']
me:  1
trying: ['load_supplier']
me:  2
trying: ['pd']
me:  0


Declaring variable tpch_parent
Declaring variable DATA_ROOT
Declaring variable STORAGE_OPTS
Declaring variable pd
Declaring variable load_partsupp
Declaring variable partsupp
Declaring variable supplier
Declaring variable load_supplier
Declaring variable load_nation
Declaring variable nation


In [4]:
%%RecordEvent
%%time
### cell 3 ###

# cell 3 optimized
# combine selection, computation, joins, and aggregation into a single GPU pipeline
df = (
    partsupp[['PS_PARTKEY','PS_SUPPKEY','PS_SUPPLYCOST','PS_AVAILQTY']]
    .assign(TOTAL_COST=lambda df: df.PS_SUPPLYCOST * df.PS_AVAILQTY)
    .merge(
        supplier[['S_SUPPKEY','S_NATIONKEY']],
        left_on='PS_SUPPKEY', right_on='S_SUPPKEY'
    )
    .merge(
        nation[nation.N_NAME == 'GERMANY'][['N_NATIONKEY']],
        left_on='S_NATIONKEY', right_on='N_NATIONKEY'
    )
)
# compute threshold on GPU
sum_val = float(df.TOTAL_COST.sum()) * 0.0001
# group, sum and filter on GPU
total = (
    df.groupby('PS_PARTKEY', sort=False)
      .TOTAL_COST.sum()
      .reset_index(name='VALUE')
)
total = total[total.VALUE > sum_val].sort_values('VALUE', ascending=False)

CPU times: user 60 ms, sys: 48.2 ms, total: 108 ms
Wall time: 119 ms


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q11/rewritten/o4_mini_high/checkpoints/post_cell_3_try_1.pickle

migration speed (bps): 791176753.5156715
---------------------------
variables to migrate:
partsupp 48
load_partsupp 144
total 48
DATA_ROOT 80
supplier 48
load_supplier 144
nation 48
STORAGE_OPTS 64
load_nation 144
pd 72
tpch_parent 54
sum_val 24
df 48
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q11/rewritten/o4_mini_high/checkpoints/post_cell_3_try_1.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables []
Intermediate variables ['partsupp']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables ['supplier']
Future variables ['partsupp']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables ['nation']
Future variables ['partsupp', 'supplier']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables []
Intermediate variables ['total', 'df', 'sum_val']
Future variables []
Modified dataframes
Saved cell execution info to opt_cell_exec_info


In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q11/opt_cell_exec_info_3_try_1.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[3], f)


In [ ]:
opt_output = Out.get(4)

In [ ]:
total_opt = total
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q11/annotated/checkpoints/post_cell_3.pickle

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output
